In [1]:
import torch
import onnx
import onnxruntime as ort
import numpy as np
from glasses_detector import GlassesClassifier

wrapper = GlassesClassifier(kind="anyglasses", size="medium")
print(wrapper.model)
model = wrapper.model
model.eval()
device = torch.device("cpu")
model.to(device)
dummy_input = torch.randn(1, 3, 256, 256).to(device)

onnx_file_path = "glasses_classifier.onnx"
torch.onnx.export(
    model,
    (dummy_input,),
    onnx_file_path,
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    }
)

onnx_model = onnx.load(onnx_file_path)
onnx.checker.check_model(onnx_model)

ort_session = ort.InferenceSession(onnx_file_path)
ort_inputs = {ort_session.get_inputs()[0].name: dummy_input.numpy()}
ort_outs = ort_session.run(None, ort_inputs)

ShuffleNetV2(
  (conv1): Sequential(
    (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (stage2): Sequential(
    (0): InvertedResidual(
      (branch1): Sequential(
        (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=24, bias=False)
        (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): Conv2d(24, 58, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (3): BatchNorm2d(58, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (4): ReLU(inplace=True)
      )
      (branch2): Sequential(
        (0): Conv2d(24, 58, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(58, eps=1e-05, momentum=0.1, affine=True, track_running_

C:\Users\Josh\AppData\Local\Temp\ipykernel_33148\1852081485.py:16: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0228 22:16:20.955000 33148 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0228 22:16:20.958000 33148 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0228 22:16:20.961000 33148 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for paramete

[torch.onnx] Obtain model graph for `ShuffleNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ShuffleNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 144 of general pattern rewrite rules.


In [2]:
import onnx
model = onnx.load("glasses_classifier.onnx")
print("Opset version:", model.opset_import[0].version)
onnx.checker.check_model(model)  # Should pass

Opset version: 18


In [3]:
import onnx

# Paths to your files (adjust if necessary)
model_path = "glasses_classifier.onnx"  # The main model file
single_file_path = "glasses_classifier_single.onnx"  # Name for your new combined file

# 1. Load the model. ONNX automatically finds the external data
#    if it's in the same folder [citation:2][citation:4][citation:8].
print(f"Loading model from {model_path}...")
model = onnx.load(model_path)

# 2. Convert the model to use embedded data.
#    This function modifies the model in-place, pulling the external weights
#    back into the model's internal structure [citation:3][citation:4].
print("Converting from external data...")
onnx.external_data_helper.convert_model_from_external_data(model)

# 3. Save the model as a single file.
#    Since the model now has internal data, onnx.save will create one file [citation:4].
print(f"Saving combined model to {single_file_path}...")
onnx.save(model, single_file_path)

print("Done! You now have a single ONNX file.")

Loading model from glasses_classifier.onnx...
Converting from external data...
Saving combined model to glasses_classifier_single.onnx...
Done! You now have a single ONNX file.
